# Bronze para Silver: Unidades Consumidoras (V-Energy Grid 360)

Execute as células em ordem, de cima para baixo.

**Entrada:** `` `v-energy`.bronze.tb_unidades_consumidoras `` (snapshot fixado com `VERSION AS OF`).
**Saída:** `` `v-energy`.silver.tb_unidades_consumidoras `` e `` `v-energy`.silver.tb_unidades_consumidoras_quarentena ``.

**Fluxo:**
1. Leitura da Bronze com versão Delta fixada (`VERSION AS OF`) e captura única de `DATA_EXECUCAO`.
2. Rejeição pré-deduplicação (quarentena de lixo, cabeçalho repetido e dados de teste).
3. Deduplicação determinística por `id_uc` (priorizando data válida, ingestão mais recente, arquivo e hash SHA-256).
4. Validação estrutural de CPF/CNPJ e montagem da quarentena unificada.
5. Padronização e tipagem (incluindo correção de encoding UTF-8/Latin-1 nos bairros e preservação das grafias canônicas de `municipio`, `bairro` e `classe` conforme `dim_uc` e `GLOSSARY.md`).
6. Validações bloqueantes e reconciliação de volumes **antes** da escrita (`df_silver` e `df_quarentena`).
7. Gravação Delta com proteção `RESTORE TABLE` / `DROP TABLE` e conferência pós-escrita.
8. Perfilamento e alertas informativos.

In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 1: Configuracao, parametros de regra e leitura da Bronze (VERSION AS OF)
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from pyspark.sql.window import Window

catalog = 'v-energy'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.silver')

bronze_table = f'`{catalog}`.bronze.tb_unidades_consumidoras'
silver_table = f'`{catalog}`.silver.tb_unidades_consumidoras'
quarantine_table = f'`{catalog}`.silver.tb_unidades_consumidoras_quarentena'

# Capturada uma unica vez no inicio para evitar divergencia se a execucao cruzar a meia-noite
DATA_EXECUCAO = str(spark.sql('SELECT current_date()').first()[0])

# Parametros de regra (documentados na arquitetura)
MIN_DIGITOS_DOCUMENTO = 9  # abaixo disso o documento e considerado incompleto
DATA_LIGACAO_MIN = '1900-01-01'  # datas fora de [MIN, DATA_EXECUCAO] viram NULL
BRASIL_LAT_MIN, BRASIL_LAT_MAX = -34.0, 6.0
BRASIL_LON_MIN, BRASIL_LON_MAX = -74.0, -28.0
CEPS_SENTINELA = ['99991231', '00000000']
UFS_VALIDAS = [
    'AC',
    'AL',
    'AP',
    'AM',
    'BA',
    'CE',
    'DF',
    'ES',
    'GO',
    'MA',
    'MT',
    'MS',
    'MG',
    'PA',
    'PB',
    'PR',
    'PE',
    'PI',
    'RJ',
    'RN',
    'RS',
    'RO',
    'RR',
    'SC',
    'SP',
    'SE',
    'TO',
]
CLASSES_VALIDAS = ['Residencial', 'Comercial', 'Industrial', 'Rural', 'Poder Público']

MOTIVOS_PRE = [
    'id_uc_nulo',
    'cabecalho_repetido',
    'dado_de_teste_id_uc',
    'nome_titular_nulo',
    'dado_de_teste_titular',
]
MOTIVOS_POS = [
    'documento_ausente',
    'documento_incompleto',
    'documento_tamanho_invalido',
    'documento_sentinela_repetido',
]


def fmt(n):
    return f'{n:,}'.replace(',', '.')


# Leitura da Bronze fixando a versao Delta atual
versao_bronze = spark.sql(f'DESCRIBE HISTORY {bronze_table} LIMIT 1').first()['version']
df_bronze = spark.sql(f'SELECT * FROM {bronze_table} VERSION AS OF {versao_bronze}')

colunas_esperadas = [
    'id_uc',
    'cpf_cnpj',
    'nome_titular',
    'logradouro',
    'numero',
    'bairro',
    'municipio',
    'uf',
    'cep',
    'classe',
    'subgrupo_tarifario',
    'alimentador',
    'data_ligacao',
    'status',
    'latitude',
    'longitude',
    'ingestion_datetime',
    'source_file',
]
faltantes = [c for c in colunas_esperadas if c not in df_bronze.columns]
assert not faltantes, f'Colunas ausentes na Bronze: {faltantes}'

total_bronze = df_bronze.count()
print(f'Origem Bronze: {bronze_table} (versão {versao_bronze})')
print(f'Destino Silver: {silver_table}')
print(f'Destino Quarentena: {quarantine_table}')
print(f'Data de referência da execução: {DATA_EXECUCAO}')
print(f'Total de registros na Bronze: {fmt(total_bronze)}')
df_bronze.printSchema()

Origem Bronze: `v-energy`.bronze.tb_unidades_consumidoras (versão 23)
Destino Silver: `v-energy`.silver.tb_unidades_consumidoras
Destino Quarentena: `v-energy`.silver.tb_unidades_consumidoras_quarentena
Data de referência da execução: 2026-11-01
Total de registros na Bronze: 46.461
root
 |-- id_uc: string (nullable = true)
 |-- cpf_cnpj: string (nullable = true)
 |-- nome_titular: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- cep: string (nullable = true)
 |-- classe: string (nullable = true)
 |-- subgrupo_tarifario: string (nullable = true)
 |-- alimentador: string (nullable = true)
 |-- data_ligacao: string (nullable = true)
 |-- status: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_datetime: t

In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 2: Funcoes auxiliares, correcao de encoding e digitos verificadores
# ==============================================================================
MOJIBAKE_PARES = [
    ('Ã\x81', 'Á'),
    ('Ã\x82', 'Â'),
    ('Ã\x93', 'Ó'),
    ('Ã¡', 'á'),
    ('Ã¢', 'â'),
    ('Ã£', 'ã'),
    ('Ã§', 'ç'),
    ('Ã©', 'é'),
    ('Ã\xad', 'í'),
    ('Ã³', 'ó'),
    ('Ã´', 'ô'),
    ('Ãµ', 'õ'),
    ('Ãº', 'ú'),
]
MAPA_CLASSE = {
    'RESIDENCIAL': 'Residencial',
    'COMERCIAL': 'Comercial',
    'INDUSTRIAL': 'Industrial',
    'RURAL': 'Rural',
    'PODER PUBLICO': 'Poder Público',
    'PODER PÚBLICO': 'Poder Público',
}
mapa_classe_expr = F.create_map(
    [item for par in MAPA_CLASSE.items() for item in (F.lit(par[0]), F.lit(par[1]))]
)


def limpar_espacos(c):
    """trim + colapsa espacos repetidos."""
    return F.regexp_replace(F.trim(c), r'\s+', ' ')


def corrigir_mojibake(c):
    """Corrige sequencias UTF-8 lidas como Latin-1 (ex.: Jardim JordÃ£o -> Jardim Jordão)."""
    res = limpar_espacos(c)
    for ruim, bom in MOJIBAKE_PARES:
        res = F.replace(res, F.lit(ruim), F.lit(bom))
    return res


def padronizar_classe(c):
    """Padroniza classe para os 5 valores canonicos do contrato dim_uc (com acento)."""
    limpa = F.upper(corrigir_mojibake(c))
    return F.coalesce(mapa_classe_expr[limpa], corrigir_mojibake(c))


def _cpf_dv_valido(d):
    if d is None or len(d) != 11 or not d.isdigit() or d == d[0] * 11:
        return False
    for n in (9, 10):
        soma = sum(int(d[i]) * (n + 1 - i) for i in range(n))
        dv = (soma * 10) % 11
        dv = 0 if dv == 10 else dv
        if dv != int(d[n]):
            return False
    return True


def _cnpj_dv_valido(d):
    if d is None or len(d) != 14 or not d.isdigit() or d == d[0] * 14:
        return False
    pesos1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    pesos2 = [6] + pesos1
    for pesos, pos in ((pesos1, 12), (pesos2, 13)):
        soma = sum(int(d[i]) * pesos[i] for i in range(pos))
        resto = soma % 11
        dv = 0 if resto < 2 else 11 - resto
        if dv != int(d[pos]):
            return False
    return True


@F.udf(returnType=BooleanType())
def udf_documento_dv_valido(doc):
    if doc is None:
        return False
    if len(doc) == 11:
        return _cpf_dv_valido(doc)
    if len(doc) == 14:
        return _cnpj_dv_valido(doc)
    return False

In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 3: Rejeicao de lixo/testes e deduplicacao deterministica
# ==============================================================================
# 3.1 Normalizacao minima das chaves usadas nas regras (as colunas originais ficam intactas)
df_base = df_bronze.withColumn('id_uc_norm', F.upper(F.trim(F.col('id_uc')))).withColumn(
    'nome_norm', limpar_espacos(F.col('nome_titular'))
)

# 3.2 Regras de rejeicao (null-safe: nulos tambem vao para a quarentena, nunca somem em silencio)
motivo_pre = (
    F.when(F.col('id_uc_norm').isNull() | (F.col('id_uc_norm') == ''), F.lit('id_uc_nulo'))
    .when(F.col('id_uc_norm') == 'ID_UC', F.lit('cabecalho_repetido'))
    .when(F.col('id_uc_norm').startswith('TESTE'), F.lit('dado_de_teste_id_uc'))
    .when(F.col('nome_norm').isNull() | (F.col('nome_norm') == ''), F.lit('nome_titular_nulo'))
    .when(F.lower(F.col('nome_norm')).isin('a', 'teste'), F.lit('dado_de_teste_titular'))
)

df_pre = df_base.withColumn('motivo_rejeicao', motivo_pre)
df_rej_pre = df_pre.filter(F.col('motivo_rejeicao').isNotNull())
df_valid_pre = df_pre.filter(F.col('motivo_rejeicao').isNull()).drop('motivo_rejeicao')

# 3.3 Data de ligacao tipada ANTES da dedup e restrita a [DATA_LIGACAO_MIN, DATA_EXECUCAO]
data_cast = F.expr('try_cast(data_ligacao as date)')
data_valida = F.when(
    (data_cast >= F.to_date(F.lit(DATA_LIGACAO_MIN)))
    & (data_cast <= F.to_date(F.lit(DATA_EXECUCAO))),
    data_cast,
)

# 3.4 Hash do conteudo de negocio: ultimo criterio de desempate (torna a dedup 100% deterministica)
colunas_negocio = [
    c for c in df_bronze.columns if c not in ('ingestion_datetime', 'source_file')
]
row_hash = F.sha2(
    F.concat_ws(
        '||', *[F.coalesce(F.col(c).cast('string'), F.lit('')) for c in colunas_negocio]
    ),
    256,
)

df_valid_pre = df_valid_pre.withColumn('data_ligacao_dt', data_valida).withColumn(
    'row_hash', row_hash
)

# 3.5 Dedup: versao mais recente por id_uc
window_spec = Window.partitionBy('id_uc_norm').orderBy(
    F.col('ingestion_datetime').desc_nulls_last(),
    F.col('data_ligacao_dt').desc_nulls_last(),
    F.col('source_file').desc_nulls_last(),
    F.col('row_hash').desc(),
)

df_dedup = (
    df_valid_pre.withColumn('row_num', F.row_number().over(window_spec))
    .filter(F.col('row_num') == 1)
    .drop('row_num')
)
print('Rejeicao e deduplicacao definidas (execucao lazy, nada foi gravado ainda).')

Rejeicao e deduplicacao definidas (execucao lazy, nada foi gravado ainda).


In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 4: Validacao estrutural de documentos e montagem da quarentena
# ==============================================================================
doc_digitos = F.regexp_replace(F.coalesce(F.col('cpf_cnpj'), F.lit('')), r'\D', '')

motivo_pos = (
    F.when(F.length('doc_digitos') == 0, F.lit('documento_ausente'))
    .when(F.length('doc_digitos') > 14, F.lit('documento_tamanho_invalido'))
    .when(F.length('doc_digitos') < MIN_DIGITOS_DOCUMENTO, F.lit('documento_incompleto'))
    .when(F.col('doc_digitos').rlike(r'^(\d)\1*$'), F.lit('documento_sentinela_repetido'))
)

df_doc = df_dedup.withColumn('doc_digitos', doc_digitos).withColumn(
    'motivo_rejeicao', motivo_pos
)

df_rej_pos = df_doc.filter(F.col('motivo_rejeicao').isNotNull())
df_ok = df_doc.filter(F.col('motivo_rejeicao').isNull())

# Quarentena: colunas originais da Bronze + motivo + timestamp
cols_bronze = df_bronze.columns
df_quarentena = (
    df_rej_pre.select(*cols_bronze, 'motivo_rejeicao')
    .unionByName(df_rej_pos.select(*cols_bronze, 'motivo_rejeicao'))
    .withColumn('quarentena_datetime', F.current_timestamp())
)
print('Validacao de documentos e quarentena definidas (lazy).')

Validacao de documentos e quarentena definidas (lazy).


In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 5: Padronizacao, tipagem e metadados da Silver
# ==============================================================================
# Documento: so digitos; <= 11 vira CPF (lpad 11), senao CNPJ (lpad 14)
doc_padronizado = F.when(
    F.length('doc_digitos') <= 11, F.lpad('doc_digitos', 11, '0')
).otherwise(F.lpad('doc_digitos', 14, '0'))

df_silver = (
    df_ok
    # Identificacao e titular
    .withColumn('id_uc', F.col('id_uc_norm'))
    .withColumn('nome_titular', F.col('nome_norm'))
    # Documento
    .withColumn('cpf_cnpj', doc_padronizado)
    .withColumn(
        'tipo_documento',
        F.when(F.length('cpf_cnpj') == 11, 'CPF').otherwise('CNPJ'),
    )
    .withColumn('documento_dv_valido', udf_documento_dv_valido(F.col('cpf_cnpj')))
    # Endereco (corrigindo mojibake UTF-8/Latin-1 nos 59 bairros e preservando acentos)
    .withColumn('logradouro', limpar_espacos(F.col('logradouro')))
    .withColumn('numero', F.trim(F.col('numero')))
    .withColumn('bairro', corrigir_mojibake(F.col('bairro')))
    .withColumn('municipio', corrigir_mojibake(F.col('municipio')))
    # UF: remove pontos/espacos (P.E. -> PE), nome por extenso de PE, dominio das 27 UFs
    .withColumn('_uf', F.regexp_replace(F.upper(F.trim(F.col('uf'))), r'[\.\s]', ''))
    .withColumn(
        '_uf',
        F.when(F.col('_uf') == 'PERNAMBUCO', F.lit('PE')).otherwise(F.col('_uf')),
    )
    .withColumn('uf', F.when(F.col('_uf').isin(*UFS_VALIDAS), F.col('_uf')))
    # CEP: so digitos, exatamente 8, sem sentinelas (99991231, 00000000) nem digitos repetidos
    .withColumn('_cep', F.regexp_replace(F.coalesce(F.col('cep'), F.lit('')), r'\D', ''))
    .withColumn(
        'cep',
        F.when(
            (F.length('_cep') == 8)
            & (~F.col('_cep').rlike(r'^(\d)\1*$'))
            & (~F.col('_cep').isin(*CEPS_SENTINELA)),
            F.col('_cep'),
        ),
    )
    # Classificacao (5 classes canonicas com acento em Poder Público)
    .withColumn('classe', padronizar_classe(F.col('classe')))
    .withColumn('subgrupo_tarifario', F.upper(limpar_espacos(F.col('subgrupo_tarifario'))))
    .withColumn('alimentador', F.upper(limpar_espacos(F.col('alimentador'))))
    .withColumn('status', F.upper(limpar_espacos(F.col('status'))))
    # Data (ja tipada e restrita na celula 3)
    .withColumn('data_ligacao', F.col('data_ligacao_dt'))
    # Coordenadas: virgula decimal aceita; par fora da caixa do Brasil vira NULL/NULL
    .withColumn('_lat', F.expr("try_cast(replace(trim(latitude), ',', '.') as double)"))
    .withColumn('_lon', F.expr("try_cast(replace(trim(longitude), ',', '.') as double)"))
    .withColumn(
        '_coord_ok',
        F.col('_lat').between(BRASIL_LAT_MIN, BRASIL_LAT_MAX)
        & F.col('_lon').between(BRASIL_LON_MIN, BRASIL_LON_MAX),
    )
    .withColumn('latitude', F.when(F.col('_coord_ok'), F.col('_lat')))
    .withColumn('longitude', F.when(F.col('_coord_ok'), F.col('_lon')))
    # Metadado Silver (ingestion_datetime e source_file vem da Bronze)
    .withColumn('silver_processed_datetime', F.current_timestamp())
)

colunas_silver = [
    'id_uc',
    'cpf_cnpj',
    'tipo_documento',
    'documento_dv_valido',
    'nome_titular',
    'logradouro',
    'numero',
    'bairro',
    'municipio',
    'uf',
    'cep',
    'classe',
    'subgrupo_tarifario',
    'alimentador',
    'data_ligacao',
    'status',
    'latitude',
    'longitude',
    'ingestion_datetime',
    'source_file',
    'silver_processed_datetime',
]
df_silver = df_silver.select(*colunas_silver)

display(df_silver.limit(10))

id_uc,cpf_cnpj,tipo_documento,documento_dv_valido,nome_titular,logradouro,numero,bairro,municipio,uf,cep,classe,subgrupo_tarifario,alimentador,data_ligacao,status,latitude,longitude,ingestion_datetime,source_file,silver_processed_datetime
UC100008,92516197000134,CNPJ,true,Auto Pecas Nascimento LTDA,Rua da Paz,670,Recife,Recife,PE,56128739,Comercial,B3,ALM-REC-035,2014-09-16,ATIVA,-8.03,-34.89,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100014,46545969412,CPF,true,Luis Melo Costa,Travessa Sao Jose,1700,Alto José Bonifácio,Recife,PE,50451291,Residencial,B1,ALM-REC-002,2004-07-26,ATIVA,-8.04,-34.92,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100016,24213803678,CPF,true,Andre Torres Ribeiro,Rua Quinze de Novembro,69,Sossego,Ilha de Itamaracá,PE,54347062,Residencial,B1,ALM-ILH-001,1903-01-01,ATIVA,-7.73,-34.8,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100020,34440300375,CPF,true,Marcia Carvalho Pinto,Travessa Agamenon Magalhaes,1331,Centro,Jaboatão dos Guararapes,PE,53812607,Residencial,B1,ALM-JAB-003,2002-09-03,ATIVA,-8.1,-35.02,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100024,40834887274,CPF,true,Antonia Santos Martins,Travessa Padre Cicero,2047,Ipsep,Recife,PE,53352710,Residencial,B1,ALM-REC-025,2002-12-08,ATIVA,-8.06,-34.89,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100029,06064472022,CPF,true,Antonia Campos Dias,Avenida Frei Caneca,941,Vila Rica,Jaboatão dos Guararapes,PE,55268824,Residencial,B1,ALM-JAB-019,2007-02-04,ATIVA,-8.1,-35.03,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100030,49353447097,CPF,true,Rafael Siqueira,Praca Setubal,909,Macaxeira,Recife,PE,52035113,Residencial,B1,ALM-REC-027,2007-02-12,ATIVA,-8.06,-34.89,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100032,03778212761,CPF,true,Manoel Correia Alves,Praca Joaquim Nabuco,2042,Engenho Velho,Jaboatão dos Guararapes,PE,55950871,Residencial,B1,ALM-JAB-005,2009-08-12,ATIVA,-8.09,-34.99,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100040,06051639012,CPF,true,Rodrigo Lima,Rua Frei Caneca,861,Iputinga,Recife,PE,56661341,Residencial,B1,ALM-REC-024,2014-03-15,ATIVA,-8.07,-34.88,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z
UC100042,29803652460,CPF,true,Alexandre Duarte Siqueira,Estrada Setubal,788,Alto José Bonifácio,Recife,PE,53461358,Residencial,B1,ALM-REC-002,2019-01-13,ATIVA,-8.04,-34.87,2026-10-31T22:15:33.890Z,unidades_consumidoras.csv,2026-11-01T00:47:17.446Z


In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 6: Reconciliacao de volumes e validacoes BLOQUEANTES (ANTES da escrita)
# ==============================================================================
def validar_uc_silver(df_s_alvo, df_q_alvo, rotulo):
    falhas = []

    def checar(nome, condicao, detalhe=''):
        print(f'[{"OK   " if condicao else "FALHA"}] ({rotulo}) {nome} {detalhe}')
        if not condicao:
            falhas.append(nome)

    n_silver = df_s_alvo.count()
    rej = {
        r['motivo_rejeicao']: r['qtd']
        for r in df_q_alvo.groupBy('motivo_rejeicao')
        .agg(F.count(F.lit(1)).alias('qtd'))
        .collect()
    }
    n_rej_pre = sum(rej.get(m, 0) for m in MOTIVOS_PRE)
    n_rej_pos = sum(rej.get(m, 0) for m in MOTIVOS_POS)

    n_valid_pre = df_valid_pre.count()
    n_ids_distintos = df_valid_pre.select('id_uc_norm').distinct().count()
    n_duplicatas = n_valid_pre - n_ids_distintos

    print(f'\n=== Reconciliacao de volumes ({rotulo}) ===')
    print(f'Bronze (total lido)                      : {fmt(total_bronze)}')
    for m in MOTIVOS_PRE:
        print(f'  (-) rejeitado: {m:<30}: {fmt(rej.get(m, 0))}')
    print(f'  (-) duplicatas descartadas (dedup)     : {fmt(n_duplicatas)}')
    for m in MOTIVOS_POS:
        print(f'  (-) rejeitado: {m:<30}: {fmt(rej.get(m, 0))}')
    print(f'Silver (validada)                        : {fmt(n_silver)}\n')

    checar(
        'motivos de quarentena conhecidos',
        set(rej) <= set(MOTIVOS_PRE + MOTIVOS_POS),
        f'{sorted(rej)}',
    )
    checar('pre-rejeicao + validos = Bronze', n_rej_pre + n_valid_pre == total_bronze)
    checar(
        'ids distintos validos = Silver + rejeitados de documento',
        n_ids_distintos == n_silver + n_rej_pos,
    )
    checar(
        'Bronze = rejeitados + duplicatas + Silver',
        total_bronze == n_rej_pre + n_duplicatas + n_rej_pos + n_silver,
    )

    g = df_s_alvo.agg(
        F.count(F.lit(1)).alias('total'),
        F.countDistinct('id_uc').alias('ids_distintos'),
        F.sum(
            F.when(
                F.col('id_uc').isNull() | ~F.col('id_uc').rlike(r'^UC\d{6}$'), 1
            ).otherwise(0)
        ).alias('id_uc_invalido'),
        F.sum(F.when(F.col('cpf_cnpj').isNull(), 1).otherwise(0)).alias('nulos_cpf_cnpj'),
        F.sum(F.when(F.col('nome_titular').isNull(), 1).otherwise(0)).alias('nulos_nome'),
        F.sum(F.when(~F.length('cpf_cnpj').isin(11, 14), 1).otherwise(0)).alias(
            'doc_tamanho_invalido'
        ),
        F.sum(
            F.when(
                (F.col('tipo_documento') == 'CPF') != (F.length('cpf_cnpj') == 11), 1
            ).otherwise(0)
        ).alias('tipo_inconsistente'),
        F.sum(
            F.when(
                F.col('cep').isNotNull()
                & ((F.length('cep') != 8) | F.col('cep').isin(*CEPS_SENTINELA)),
                1,
            ).otherwise(0)
        ).alias('cep_invalido'),
        F.sum(
            F.when(F.col('uf').isNotNull() & ~F.col('uf').isin(*UFS_VALIDAS), 1).otherwise(0)
        ).alias('uf_invalida'),
        F.sum(
            F.when(
                F.col('classe').isNull() | ~F.col('classe').isin(*CLASSES_VALIDAS), 1
            ).otherwise(0)
        ).alias('classe_invalida'),
        F.sum(
            F.when(F.col('bairro').isNull() | F.col('bairro').contains('Ã'), 1).otherwise(0)
        ).alias('bairro_mojibake'),
        F.sum(F.when(F.col('id_uc').startswith('TESTE'), 1).otherwise(0)).alias('ids_teste'),
    ).collect()[0]

    checar('Silver nao vazia', g['total'] > 0)
    checar(
        'id_uc unico',
        g['total'] == g['ids_distintos'],
        f'(total={fmt(g["total"])}, distintos={fmt(g["ids_distintos"])})',
    )
    checar('id_uc no padrao UC + 6 digitos', (g['id_uc_invalido'] or 0) == 0)
    checar('sem nulos em cpf_cnpj', (g['nulos_cpf_cnpj'] or 0) == 0)
    checar('sem nulos em nome_titular', (g['nulos_nome'] or 0) == 0)
    checar('cpf_cnpj com 11 ou 14 digitos', (g['doc_tamanho_invalido'] or 0) == 0)
    checar('tipo_documento consistente com o tamanho', (g['tipo_inconsistente'] or 0) == 0)
    checar('cep valido ou NULL', (g['cep_invalido'] or 0) == 0)
    checar('uf valida ou NULL', (g['uf_invalida'] or 0) == 0)
    checar('classe dentro dos 5 valores canonicos', (g['classe_invalida'] or 0) == 0)
    checar('bairro sem encoding corrompido (mojibake)', (g['bairro_mojibake'] or 0) == 0)
    checar('sem id_uc de teste', (g['ids_teste'] or 0) == 0)

    if falhas:
        raise RuntimeError(f'Validacoes bloqueantes falharam ({rotulo}): {falhas}')
    print(f'\nTodas as validacoes bloqueantes ({rotulo}) passaram.')
    return n_silver, sum(rej.values())


n_silver_esperado, n_quar_esperado = validar_uc_silver(
    df_silver, df_quarentena, 'antes de gravar'
)


=== Reconciliacao de volumes (antes de gravar) ===
Bronze (total lido)                      : 46.461
  (-) rejeitado: id_uc_nulo                    : 0
  (-) rejeitado: cabecalho_repetido            : 1
  (-) rejeitado: dado_de_teste_id_uc           : 200
  (-) rejeitado: nome_titular_nulo             : 0
  (-) rejeitado: dado_de_teste_titular         : 0
  (-) duplicatas descartadas (dedup)     : 1.260
  (-) rejeitado: documento_ausente             : 0
  (-) rejeitado: documento_incompleto          : 0
  (-) rejeitado: documento_tamanho_invalido    : 0
  (-) rejeitado: documento_sentinela_repetido  : 0
Silver (validada)                        : 45.000

[OK   ] (antes de gravar) motivos de quarentena conhecidos ['cabecalho_repetido', 'dado_de_teste_id_uc']
[OK   ] (antes de gravar) pre-rejeicao + validos = Bronze 
[OK   ] (antes de gravar) ids distintos validos = Silver + rejeitados de documento 
[OK   ] (antes de gravar) Bronze = rejeitados + duplicatas + Silver 
[OK   ] (antes de gr

In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 7: Gravacao da Silver e Quarentena com protecao RESTORE e conferencia
# ==============================================================================
def versao_atual(tabela):
    if not spark.catalog.tableExists(tabela):
        return None
    return spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()['version']


def restaurar_tabela(tabela, versao_anterior):
    try:
        if not spark.catalog.tableExists(tabela):
            return
        if versao_anterior is None:
            spark.sql(f'DROP TABLE IF EXISTS {tabela}')
        else:
            spark.sql(f'RESTORE TABLE {tabela} TO VERSION AS OF {versao_anterior}')
    except Exception as exc:
        print(f'[ERRO] falha ao restaurar {tabela}: {exc}')


versao_ant_silver = versao_atual(silver_table)
versao_ant_quar = versao_atual(quarantine_table)

try:
    (
        df_silver.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(silver_table)
    )
    (
        df_quarentena.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(quarantine_table)
    )
    df_silver_final = spark.table(silver_table)
    df_quar_final = spark.table(quarantine_table)
    if df_silver_final.count() != n_silver_esperado:
        raise RuntimeError(
            f'Silver gravada com {df_silver_final.count()} linhas, '
            f'esperado {n_silver_esperado}.'
        )
    if df_quar_final.count() != n_quar_esperado:
        raise RuntimeError(
            f'Quarentena gravada com {df_quar_final.count()} linhas, '
            f'esperado {n_quar_esperado}.'
        )
    validar_uc_silver(df_silver_final, df_quar_final, 'tabela gravada')
except Exception:
    restaurar_tabela(silver_table, versao_ant_silver)
    restaurar_tabela(quarantine_table, versao_ant_quar)
    raise

print(f'Silver gravada: {silver_table} ({fmt(n_silver_esperado)} linhas)')
print(f'Quarentena gravada: {quarantine_table} ({fmt(n_quar_esperado)} linhas)')

# Constraints e comentario (idempotentes). Falha aqui gera aviso; o gate real e validar_uc_silver.
try:
    for coluna in ['id_uc', 'cpf_cnpj', 'nome_titular']:
        spark.sql(f'ALTER TABLE {silver_table} ALTER COLUMN {coluna} SET NOT NULL')
    spark.sql(f'ALTER TABLE {silver_table} DROP CONSTRAINT IF EXISTS chk_cpf_cnpj_tamanho')
    spark.sql(
        f'ALTER TABLE {silver_table} '
        'ADD CONSTRAINT chk_cpf_cnpj_tamanho CHECK (length(cpf_cnpj) IN (11, 14))'
    )
    spark.sql(
        f'COMMENT ON TABLE {silver_table} IS '
        "'Silver de Unidades Consumidoras: 1 linha por id_uc, higienizada e tipada.'"
    )
    print('Constraints aplicadas (NOT NULL em id_uc, cpf_cnpj, nome_titular; CHECK documento).')
except Exception as e:
    print(f'AVISO: nao foi possivel aplicar constraints/comentario: {e}')


=== Reconciliacao de volumes (tabela gravada) ===
Bronze (total lido)                      : 46.461
  (-) rejeitado: id_uc_nulo                    : 0
  (-) rejeitado: cabecalho_repetido            : 1
  (-) rejeitado: dado_de_teste_id_uc           : 200
  (-) rejeitado: nome_titular_nulo             : 0
  (-) rejeitado: dado_de_teste_titular         : 0
  (-) duplicatas descartadas (dedup)     : 1.260
  (-) rejeitado: documento_ausente             : 0
  (-) rejeitado: documento_incompleto          : 0
  (-) rejeitado: documento_tamanho_invalido    : 0
  (-) rejeitado: documento_sentinela_repetido  : 0
Silver (validada)                        : 45.000

[OK   ] (tabela gravada) motivos de quarentena conhecidos ['cabecalho_repetido', 'dado_de_teste_id_uc']
[OK   ] (tabela gravada) pre-rejeicao + validos = Bronze 
[OK   ] (tabela gravada) ids distintos validos = Silver + rejeitados de documento 
[OK   ] (tabela gravada) Bronze = rejeitados + duplicatas + Silver 
[OK   ] (tabela gravada) 

In [0]:
# ruff: noqa: F821
# ==============================================================================
# CELULA 8: Perfilamento e validacoes de ALERTA (nao bloqueiam o pipeline)
# ==============================================================================
df_s = spark.table(silver_table)
total = df_s.count()

# 8.1 Nulos em colunas opcionais (apos as regras de saneamento)
print('=== Nulos em colunas opcionais ===')
colunas_op = [
    'data_ligacao',
    'latitude',
    'longitude',
    'cep',
    'uf',
    'classe',
    'municipio',
    'bairro',
    'logradouro',
    'status',
    'alimentador',
    'subgrupo_tarifario',
]
for c in colunas_op:
    n = df_s.filter(F.col(c).isNull()).count()
    marca = 'ALERTA' if n > 0 else 'ok    '
    print(f'[{marca}] {c:<20}: {fmt(n)} ({(100.0 * n / total):.2f}%)')

# 8.2 Digito verificador de CPF/CNPJ (nao bloqueante: a base pode ser sintetica/anonimizada)
print('\n=== Digito verificador de CPF/CNPJ ===')
display(
    df_s.groupBy('tipo_documento', 'documento_dv_valido')
    .count()
    .orderBy('tipo_documento', 'documento_dv_valido')
)

# 8.3 Dominios (para decidir listas de valores validos)
print('\n=== Dominios observados ===')
for c in ['status', 'classe', 'uf', 'subgrupo_tarifario']:
    print(f'-- {c}')
    df_s.groupBy(c).count().orderBy(F.col('count').desc()).show(30, truncate=False)

# 8.4 Nomes suspeitos (digitos so sao suspeitos para CPF; razao social pode ter numeros)
nomes = df_s.select('id_uc', 'cpf_cnpj', 'tipo_documento', 'nome_titular')
susp = nomes.filter(
    (F.length('nome_titular') <= 3)
    | (~F.col('nome_titular').rlike(' '))
    | ((F.col('tipo_documento') == 'CPF') & F.col('nome_titular').rlike('[0-9]'))
)
n_susp = susp.count()
print(f'\nNomes potencialmente suspeitos: {fmt(n_susp)}')
if n_susp > 0:
    display(susp.orderBy(F.length('nome_titular')).limit(20))

# 8.5 Informativo: documentos com mais de uma UC (titular com varias UCs e comum em energia)
n_docs = df_s.select('cpf_cnpj').distinct().count()
n_docs_multi = (
    df_s.groupBy('cpf_cnpj')
    .agg(F.count(F.lit(1)).alias('qtd_ucs'))
    .filter(F.col('qtd_ucs') > 1)
    .count()
)
print(
    f'\nUCs: {fmt(total)} | documentos distintos: {fmt(n_docs)} | '
    f'documentos com mais de uma UC: {fmt(n_docs_multi)}'
)
if n_docs_multi == 0:
    print(
        'Relacao 1:1 entre UC e documento. Para uma distribuidora real isso e atipico: '
        'confirmar se a base e sintetica.'
    )

# 8.6 Quarentena por motivo
print('\n=== Quarentena por motivo ===')
spark.table(quarantine_table).groupBy('motivo_rejeicao').count().orderBy(
    F.col('count').desc()
).show(truncate=False)

=== Nulos em colunas opcionais ===
[ok    ] data_ligacao        : 0 (0.00%)
[ok    ] latitude            : 0 (0.00%)
[ok    ] longitude           : 0 (0.00%)
[ALERTA] cep                 : 1.350 (3.00%)
[ok    ] uf                  : 0 (0.00%)
[ok    ] classe              : 0 (0.00%)
[ok    ] municipio           : 0 (0.00%)
[ok    ] bairro              : 0 (0.00%)
[ok    ] logradouro          : 0 (0.00%)
[ok    ] status              : 0 (0.00%)
[ok    ] alimentador         : 0 (0.00%)
[ok    ] subgrupo_tarifario  : 0 (0.00%)

=== Digito verificador de CPF/CNPJ ===


tipo_documento,documento_dv_valido,count
CNPJ,true,6008
CPF,true,38992



=== Dominios observados ===
-- status
+------+-----+
|status|count|
+------+-----+
|ATIVA |45000|
+------+-----+

-- classe
+-------------+-----+
|classe       |count|
+-------------+-----+
|Residencial  |38336|
|Comercial    |4909 |
|Industrial   |877  |
|Rural        |656  |
|Poder Público|222  |
+-------------+-----+

-- uf
+---+-----+
|uf |count|
+---+-----+
|PE |45000|
+---+-----+

-- subgrupo_tarifario
+------------------+-----+
|subgrupo_tarifario|count|
+------------------+-----+
|B1                |38336|
|B3                |5131 |
|A4                |877  |
|B2                |656  |
+------------------+-----+


Nomes potencialmente suspeitos: 0

UCs: 45.000 | documentos distintos: 45.000 | documentos com mais de uma UC: 0
Relacao 1:1 entre UC e documento. Para uma distribuidora real isso e atipico: confirmar se a base e sintetica.

=== Quarentena por motivo ===
+-------------------+-----+
|motivo_rejeicao    |count|
+-------------------+-----+
|dado_de_teste_id_uc|200  |
|c